# Step 8 · Score applicable pathogen features

Score every question in the saved host/pathogen trait profile against every accepted pathogen–host abstract from the run. Pathogen, host, and pair scope are reflected in Jev's question state and decision cache. The step resumes from its feature decision ledger and fails visibly when scoring is incomplete.

In [7]:
import sys, json, yaml
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, staging_root_for_config
from pathogen_atlas_v4.pubmed_client import literature_cache_key
from pathogen_atlas_v4.feature_screening import build_feature_screen_tasks
from pathogen_atlas_v4.decision_runner import run_decisions
from pathogen_atlas_v4.jev_pathogen_screen import OpenRouterJevClient, make_jev_feature_screen_handler, openrouter_key_available
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('08_score_feature_profiles', RUN_ID)
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_CONFIG = json.loads((RUN_DIR / 'resolved_config.json').read_text(encoding='utf-8'))
CONFIG = RUN_CONFIG['source_config']
STAGING_TAXONOMY_ROOT = staging_root_for_config(ROOT, CONFIG) / 'taxonomy'
STAGING_ROOT = STAGING_TAXONOMY_ROOT.parent
PATHOGENS = pd.read_parquet(RUN_DIR / 'taxonomy/pathogens.parquet')
HOSTS = pd.read_parquet(STAGING_TAXONOMY_ROOT / 'hosts.parquet')
PAIR_SELECTION = pd.read_parquet(RUN_DIR / 'pathogen_host_selection.parquet')
PROFILE_DIR = RUN_DIR / 'feature_profiles'
PROFILES = [{'resolved_profile': json.loads(path.read_text(encoding='utf-8'))} for path in sorted(PROFILE_DIR.glob('*.json'))]
PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
CACHE_KEY = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
    pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
    taxonomy_snapshot=STAGING_ROOT.name,
    host_expansion_hash=CONFIG['acquisition'].get('host_expansion_hash', ''))
CACHE_DIR = ROOT / 'data/v4/cache/literature' / CACHE_KEY
SAMPLE_MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
if not SAMPLE_MATCHES_PATH.exists() or not ARTICLES_PATH.exists():
    raise FileNotFoundError(f'No completed pair sample cache for this run: {CACHE_DIR}. Run notebook 03, then notebook 04.')
SAMPLE_MATCHES = pd.read_parquet(SAMPLE_MATCHES_PATH)
ARTICLES = pd.read_parquet(ARTICLES_PATH)
SELECTED_PAIRS = set(zip(PAIR_SELECTION.pathogen_id.astype(str), PAIR_SELECTION.host_id.astype(str)))
SAMPLE_MATCHES = SAMPLE_MATCHES.loc[[key in SELECTED_PAIRS for key in zip(SAMPLE_MATCHES.pathogen_id.astype(str), SAMPLE_MATCHES.host_id.astype(str))]].copy()
print('Run:', RUN_ID, '| cache:', CACHE_KEY, '| profiles:', len(PROFILES), '| selected sample rows:', len(SAMPLE_MATCHES))
LOG.info('Feature score inputs: sample_rows=%d profiles=%d cache=%s', len(SAMPLE_MATCHES), len(PROFILES), CACHE_KEY)

2026-11-04 07:21 INFO Notebook session started at 2026-11-04T07:21:41.027680+00:00
2026-11-04 07:21 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-04 07:21 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/08_score_feature_profiles.log


2026-11-04 07:21 INFO Feature score inputs: sample_rows=40028 profiles=1 cache=4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961


Run: human-parasites-2026-11-03 | cache: 4b73154df150c014ba44424af6dc1ee25c67ac38132cc5f7caaa74728d87a961 | profiles: 1 | selected sample rows: 40028


## Build the complete applicable task set

Only accepted pathogen–host evidence abstracts enter feature scoring. Each resolved question produces a task for every accepted abstract. Missing abstracts are not scored and are excluded from the denominator.

In [8]:
FEATURE_PROFILE_VERSION = str(CONFIG.get('features', {}).get('profile_version', ''))
MODEL_VERSION = str(CONFIG.get('decisions', {}).get('model_version', 'typesafe/jev-1.14'))
FEATURE_TASKS = build_feature_screen_tasks(SAMPLE_MATCHES, ARTICLES, PAIR_SELECTION, PATHOGENS, HOSTS,
    PROFILES, PROMPTS, run_id=RUN_ID,
    run_config_version=str(RUN_CONFIG.get('config_version', CONFIG.get('run', {}).get('config_version', ''))),
    model_version=MODEL_VERSION, profile_version=FEATURE_PROFILE_VERSION)
if not FEATURE_TASKS:
    if PAIR_SELECTION.selection_status.astype(str).eq('accepted').any():
        raise RuntimeError('Accepted pathogen–host pairs exist, but no applicable feature tasks were produced. Check profiles and accepted evidence PMID samples.')
    print('No accepted pathogen–host evidence is available to score.')
else:
    coverage = pd.DataFrame(FEATURE_TASKS).groupby(['pathogen_id','host_id','feature_id'], dropna=False).pmid.nunique().rename('abstracts').reset_index()
    print('Feature tasks:', f'{len(FEATURE_TASKS):,}', '| accepted pairs:', PAIR_SELECTION.selection_status.astype(str).eq('accepted').sum(), '| features:', coverage.feature_id.nunique())
    display(coverage.groupby('host_id').agg(features=('feature_id','nunique'), pair_features=('pathogen_id','nunique'), tasks=('abstracts','sum')))
    LOG.info('Built complete applicable feature tasks=%d', len(FEATURE_TASKS))

Feature tasks: 453,924 | accepted pairs: 348 | features: 81


,features,pair_features,tasks
host_id,,,
ncbi:9606,77,348,431508


2026-11-04 07:21 INFO Built complete applicable feature tasks=453924


## Score with Jev and verify completion

The checkpoint ledger makes this resumable. Worker concurrency follows `logging.decision_workers`; completed decisions are reused on later runs with the same scoped question and abstract. Progress is reported every 5,000 completed decisions with throughput and an approximate ETA. Start and progress lines separate this run's completed tasks from cached decisions and show the overall total. When response usage is available, progress also estimates run cost and remaining OpenRouter credits from observed spend. A management key is needed for the starting credit balance; without it, cost is still reported but the remaining-credit estimate is unavailable. Progress is written to the run log.


In [9]:
FEATURE_LEDGER_PATH = RUN_DIR / 'feature_screen_ledger.parquet'
FEATURE_RAW_PATH = RUN_DIR / 'feature_screen_raw_responses.jsonl'
if FEATURE_TASKS:
    if not openrouter_key_available():
        raise RuntimeError('Feature scoring requires OPENROUTER_API_KEY or an OpenRouter Keychain entry.')
    from importlib import reload
    import pathogen_atlas_v4.decision_runner as decision_runner
    reload(decision_runner)
    LOG.info('Feature scoring queued tasks=%d', len(FEATURE_TASKS))
    JEV_CLIENT = OpenRouterJevClient(model=MODEL_VERSION)
    STARTING_CREDITS = JEV_CLIENT.get_remaining_credits()
    FEATURE_LEDGER = decision_runner.run_decisions(FEATURE_TASKS, make_jev_feature_screen_handler(model=MODEL_VERSION, client=JEV_CLIENT),
        ledger_path=FEATURE_LEDGER_PATH, raw_response_path=FEATURE_RAW_PATH,
        max_attempts=int(CONFIG.get('logging', {}).get('max_attempts', 2)),
        max_workers=max(1, int(CONFIG.get('logging', {}).get('decision_workers', 1024))),
        checkpoint_every=20, progress_every=500, starting_credit_balance_usd=STARTING_CREDITS,
        progress_callback=LOG.info)
    task_keys = {task['cache_key'] for task in FEATURE_TASKS}
    current = FEATURE_LEDGER.loc[FEATURE_LEDGER.cache_key.astype(str).isin(task_keys)]
    completed = current.loc[current.status.astype(str).eq('completed')]
    missing = task_keys - set(completed.cache_key.astype(str))
    if missing:
        raise RuntimeError(f'Feature scoring incomplete: {len(missing)} of {len(task_keys)} applicable questions remain unscored.')
    print('All applicable feature questions scored:', f'{len(completed):,}', '| ledger:', FEATURE_LEDGER_PATH)
    LOG.info('Feature scoring complete tasks=%d unique_completed=%d', len(FEATURE_TASKS), len(completed))
else:
    FEATURE_LEDGER = pd.read_parquet(FEATURE_LEDGER_PATH) if FEATURE_LEDGER_PATH.exists() else pd.DataFrame()

2026-11-04 07:22 INFO Decision screening: 97,921 pending; 355,922 cached; 453,843 total; workers=1024. OpenRouter credits at start: $9.9931.
2026-11-04 07:35 INFO Decision progress: 5,000/97,921 this run (5.1%) | cached 355,922 | overall 360,922/453,843 (79.5%) | 6.2 tasks/s | elapsed 13m 27s | ETA ~4h 10m | reported cost $0.2010; projected run cost $3.9371; predicted credits left $6.0560.
2026-11-04 07:48 INFO Decision progress: 10,000/97,921 this run (10.2%) | cached 355,922 | overall 365,922/453,843 (80.6%) | 6.3 tasks/s | elapsed 26m 25s | ETA ~3h 52m | reported cost $0.3894; projected run cost $3.8129; predicted credits left $6.1801.
2026-11-04 08:01 INFO Decision progress: 15,000/97,921 this run (15.3%) | cached 355,922 | overall 370,922/453,843 (81.7%) | 6.3 tasks/s | elapsed 39m 40s | ETA ~3h 39m | reported cost $0.5829; projected run cost $3.8051; predicted credits left $6.1879.
2026-11-04 08:15 INFO Decision progress: 20,000/97,921 this run (20.4%) | cached 355,922 | overall 

All applicable feature questions scored: 453,843 | ledger: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/human-parasites-2026-11-03/feature_screen_ledger.parquet
